### 에이전트(툴)
- llm문제:학습한 이후의 사건이나 사실에 대한 정보(할루시네이션)
이의 해결책으로 나온것
- 에이전트(툴): 위키피디아 등 다른 웹의 소스를 이용
- https://docs.langchain.com/oss/python/integrations/tools

### llm 질문 중 연산이 필요한 경우에 사용하는 에이전트

In [3]:
%pip install numexpr

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_community.agent_toolkits.load_tools import load_tools
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_community.tools import DuckDuckGoSearchRun 
import requests

In [5]:
@tool
def get_weather(latitude, longitude) -> str:
    """특정 지역 위도,경도의 현재 날씨 정보를 반환합니다."""

    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m")
    data = response.json()
    return data['current']['temperature_2m']


model = ChatOpenAI(model="gpt-4o-mini")

# 에이전트 생성
# 모델과 도구 리스트, 그리고 에이전트의 역할을 정의하는 시스템 프롬프트를 전달합니다.
tools = [get_weather]

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="당신은 유능한 비서입니다. 사용자의 질문에 도구를 사용하여 정확하게 답하세요."
)

# 5. 에이전트 실행
response = agent.invoke(
    {"messages": [{"role": "user", "content": "서울 날씨 어때?"}]}
)
# response = agent.invoke(
#     {"messages": [ HumanMessage(content="서울 날씨 어때?")]}
# )
# 마지막 메시지의 내용(content)만 출력
print(response["messages"][-1].content)
# # 결과 출력
for message in response["messages"]:
    message.pretty_print()
    print(message.content)
    print("-----")

현재 서울의 기온은 -3.7도입니다. 추가적인 날씨 정보가 필요하시면 말씀해 주세요!
================================ Human Message =================================

서울 날씨 어때?
서울 날씨 어때?
-----
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_ooEJL8Z81RiVp5Dsjhzm6kuG)
 Call ID: call_ooEJL8Z81RiVp5Dsjhzm6kuG
  Args:
    latitude: 37.5665
    longitude: 126.978

-----
================================= Tool Message =================================
Name: get_weather

-3.7
-3.7
-----
================================== Ai Message ==================================

현재 서울의 기온은 -3.7도입니다. 추가적인 날씨 정보가 필요하시면 말씀해 주세요!
현재 서울의 기온은 -3.7도입니다. 추가적인 날씨 정보가 필요하시면 말씀해 주세요!
-----


In [6]:
response

{'messages': [HumanMessage(content='서울 날씨 어때?', additional_kwargs={}, response_metadata={}, id='ea59d7c7-b006-47b2-8f3c-51db12c39be6'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 85, 'total_tokens': 108, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_29330a9688', 'id': 'chatcmpl-CzYts9tLMXnkpzQT3gpWQks0vCNeX', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019bd3fe-0c08-7121-9003-d7df444c2e9a-0', tool_calls=[{'name': 'get_weather', 'args': {'latitude': 37.5665, 'longitude': 126.978}, 'id': 'call_ooEJL8Z81RiVp5Dsjhzm6kuG', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 85, 

In [7]:
"구하려는 도시의 INNA Timezone 문자열(예 : 'Asia/Seoul') "

"구하려는 도시의 INNA Timezone 문자열(예 : 'Asia/Seoul') "

In [8]:
import pytz
from datetime import datetime

@tool
def get_time(city):
    """구하려는 도시의 INNA Timezone 문자열(예 : 'Asia/Seoul')"""

    try:
        tz = pytz.timezone(city)
        now = datetime.now(tz)
        return now.strftime(f"%Y년 %m월 %d일 %H시 %M분 %S초")
    except Exception as e:
        return f"시간을 알 수 없습니다: {city}"


model = ChatOpenAI(model="gpt-4o-mini")

# 에이전트 생성
# 모델과 도구 리스트, 그리고 에이전트의 역할을 정의하는 시스템 프롬프트를 전달합니다.
tools = [get_time, get_weather]

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="당신은 유능한 비서입니다. 사용자의 질문에 도구를 사용하여 정확하게 답하세요."
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "뉴욕의 현재시간 및 날씨 알려줘"}]}
)

print(response["messages"][-1].content)

현재 뉴욕의 시간은 2026년 1월 18일 21시 03분 25초입니다. 날씨는 -0.3도입니다.


In [9]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

tools = load_tools(["wikipedia", "llm-math"], llm=llm)
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Python310\\lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=3, lang='en', load_all_available_meta=False, doc_content_chars_max=4000)),
 Tool(name='Calculator', description='Useful for when you need to answer questions about math.', func=<bound method Chain.run of LLMMathChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='Translate a math problem into a expression that can be executed using Python\'s numexpr library. Use the output of running this code to answer the question.\n\nQuestion: ${{Question with math problem.}}\n```text\n${{single line mathematical expression that solves the problem}}\n```\n...numexpr.evaluate(text)...\n```output\n${{Output of running the code}}\n```\nAnswer: ${{Answer}}\n\nBegin.\n\nQuestion: What is 37593 * 67?\n```text\n37593 * 67\n```\n...numexpr.evalua

In [10]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

tools = load_tools(["wikipedia", "llm-math"], llm=llm)

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="사용자의 질문에 도구를 사용하여 정확하게 답하세요."
)
question = "그룹 퀸의 리더가 태어난 해는? 2026년 현재는 몇 살?"

response = agent.invoke(
    {"messages": [{"role": "user", "content": question}]}
)
# print(response)
for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

그룹 퀸의 리더가 태어난 해는? 2026년 현재는 몇 살?
================================== Ai Message ==================================
Tool Calls:
  wikipedia (call_2gZcIzPhnjQUZb3kggMflg1R)
 Call ID: call_2gZcIzPhnjQUZb3kggMflg1R
  Args:
    query: Queen band leader birth year
================================= Tool Message =================================
Name: wikipedia

Page: Queen Camilla
Summary: Camilla (born Camilla Rosemary Shand, later Parker Bowles, 17 July 1947) is Queen of the United Kingdom and 14 other Commonwealth realms as the wife of King Charles III.
Camilla was raised in East Sussex and South Kensington in England and educated in England, Switzerland and France. In 1973, she married British Army officer Andrew Parker Bowles; they divorced in 1995. Camilla and Charles were romantically involved periodically, both before and during each of their first marriages. Their relationship was highly publicised in th

In [11]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

@tool
def get_weather(location: str) -> str:
    """특정 지역의 현재 날씨 정보를 반환합니다."""
    # 실제로는 API를 호출하겠지만, 예시를 위해 고정값을 반환합니다.
    return f"{location}의 날씨는 맑음, 기온은 25도입니다."

tools = load_tools(["wikipedia", "llm-math"], llm=llm)
tools.append( get_weather ) # [WikipediaQueryRun, Tool, get_weather]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="사용자의 질문에 도구를 사용하여 정확하게 답하세요."
)
# question = "그룹 퀸의 리더가 태어난 해는? 2026년 현재는 몇 살?"
question = "서울 날씨 어때"
# response = agent.invoke(
#     {"messages": [{"role": "user", "content": question}]}
# )
input_messages = [HumanMessage(content="서울 날씨 어때?")]
response = agent.invoke(   {"messages":input_messages})

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

서울 날씨 어때?
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_RkT9N0xZJ55F5DibNRJfskEq)
 Call ID: call_RkT9N0xZJ55F5DibNRJfskEq
  Args:
    location: 서울
================================= Tool Message =================================
Name: get_weather

서울의 날씨는 맑음, 기온은 25도입니다.
================================== Ai Message ==================================

서울의 날씨는 맑고, 기온은 25도입니다.


In [12]:
from langchain_community.agent_toolkits.load_tools import get_all_tool_names
get_all_tool_names()

['sleep',
 'wolfram-alpha',
 'google-search',
 'google-search-results-json',
 'searx-search-results-json',
 'bing-search',
 'metaphor-search',
 'ddg-search',
 'google-books',
 'google-lens',
 'google-serper',
 'google-scholar',
 'google-finance',
 'google-trends',
 'google-jobs',
 'google-serper-results-json',
 'searchapi',
 'searchapi-results-json',
 'serpapi',
 'dalle-image-generator',
 'twilio',
 'searx-search',
 'merriam-webster',
 'wikipedia',
 'arxiv',
 'golden-query',
 'pubmed',
 'human',
 'awslambda',
 'stackexchange',
 'sceneXplain',
 'graphql',
 'openweathermap-api',
 'dataforseo-api-search',
 'dataforseo-api-search-json',
 'eleven_labs_text2speech',
 'google_cloud_texttospeech',
 'read_file',
 'reddit_search',
 'news-api',
 'tmdb-api',
 'podcast-api',
 'memorize',
 'llm-math',
 'open-meteo-api',
 'requests',
 'requests_get',
 'requests_post',
 'requests_patch',
 'requests_put',
 'requests_delete',
 'terminal']

In [13]:
%pip install ddgs

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

tools = load_tools(["ddg-search", "llm-math"], llm=llm)
tools

[DuckDuckGoSearchRun(api_wrapper=DuckDuckGoSearchAPIWrapper(region='wt-wt', safesearch='moderate', time='y', max_results=5, backend='auto', source='text')),
 Tool(name='Calculator', description='Useful for when you need to answer questions about math.', func=<bound method Chain.run of LLMMathChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='Translate a math problem into a expression that can be executed using Python\'s numexpr library. Use the output of running this code to answer the question.\n\nQuestion: ${{Question with math problem.}}\n```text\n${{single line mathematical expression that solves the problem}}\n```\n...numexpr.evaluate(text)...\n```output\n${{Output of running the code}}\n```\nAnswer: ${{Answer}}\n\nBegin.\n\nQuestion: What is 37593 * 67?\n```text\n37593 * 67\n```\n...numexpr.evaluate("37593 * 67")...\n```output\n2518731\n```\nAnswer: 2518731\n\nQuestion: 37593^(

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ddg-search는 실시간 정보에 강함
tools = load_tools(["ddg-search", "llm-math"], llm=llm) # 덕덕고를 하게 되면 위키피디아 대신 google search를 함

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="사용자의 질문에 도구를 사용하여 정확하게 답하세요."
)
# question = "그룹 퀸의 리더가 태어난 해는? 2026년 현재는 몇 살?"
question = "현재 프랑스 대통령은?"

response = agent.invoke(
    {"messages": [{"role": "user", "content": question}]}
)
# print(response)
for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

현재 프랑스 대통령은?
================================== Ai Message ==================================
Tool Calls:
  duckduckgo_search (call_Zsr3aA0ww86gjwF2iCm9zZnB)
 Call ID: call_Zsr3aA0ww86gjwF2iCm9zZnB
  Args:
    query: 현재 프랑스 대통령
================================= Tool Message =================================
Name: duckduckgo_search

나폴레옹 1세의 조카는 1848년 프랑스 대선을 통해 큰 표차로 초대 프랑스 대통령으로 선출되었다.t. e. 프랑스 공화국 대통령 . 프랑스 대통령 로고 프랑스 공화국 대통령 . 프랑스 공화국 대통령 Président de la République française President of the French Republic. 마크롱. 현직. 기자, BBC News. Reporting from 프랑스 파리. 프랑스 유권자들은 전통적으로 ‘실용적인 투표(le vote utile)’ 방식을 취해왔다. 프랑스 {불란서(佛蘭西)} 공화국. République française.민주운동(MoDem, 제2야당) 42석. 사회당(PS. 현재 제3야당이며, 사회민주주의를 표방하는 중도좌파정당.) 〈기자〉 프랑스 정부가 저출산 대책으로 출산휴가를 대폭 늘리는 방안을 추진합니다. 여성은 산전 6주와 산후 10주 출산 휴가를 쓸 수 있는데...
================================== Ai Message ==================================

현재 프랑스 대통령은 에마뉘엘 마크롱(Emmanuel Macro

In [16]:
search = DuckDuckGoSearchRun()
search.invoke("Obama's first name?")

"Obama promoted inclusion for LGBT Americans, becoming the first sitting U. S . president to publicly support same-sex marriage. англ. Barack Hussein Obama II[ 1 ]. Отец. Барак Хуссейн Обама — старший. Obama ' s First Retrospective Job Approval Rating Is 63%. Институт Гэллапа (англ.). Obama ’ s father, also named Barack Hussein Obama , grew up in a small village in Nyanza Province, Kenya, as a member of the Luo ethnicity. He won a scholarship to study economics at the... Obama Sr. married three times and had children by four women. With his first wife Kezia, Obama Sr. had a son and a daughter, Roy and Auma, before Barack was born. Obama ' s first name trivia, fun quiz questions, history quiz online, guess the president's name, presidential trivia questions, quiz about Obama , educational quizzes for adults..."

- DuckDuckGoSearch: 실시간 정보, 최신 뉴스, 현재 사건에 적합
- Wikipedia: 정적 지식, 인물, 역사, 개념 설명에 적합

In [20]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
mytools = load_tools(["wikipedia", "llm-math"], llm=llm)
# mytools = load_tools(["wikipedia", "ddg-search","llm-math"], llm=llm)
print('tool', mytools)
search = DuckDuckGoSearchRun(description="반드시 실시간 정보, 최신 뉴스, 현재 사건, 오늘 기준 정보가 필요할 때만 사용")
tools = [search] + mytools
# mytools = [WikipediaQueryRun(), Tool]
# mytools.append( search )
# mytools = [WikipediaQueryRun(), Tool, DuckDuckGoSearchRun()]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="사용자의 질문에 도구를 사용하여 정확하게 답하세요."
)

# question = "그룹 퀸의 리더가 태어난 해는? 2026년 현재는 몇 살?" # 위키피디아 서치
question = "최신뉴스 top5 알려줄래" # 덕덕고 서치

input_messages = [HumanMessage(content=question)]
response = agent.invoke(   {"messages":input_messages})

for message in response["messages"]:
    message.pretty_print()

tool [WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Python310\\lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=3, lang='en', load_all_available_meta=False, doc_content_chars_max=4000)), Tool(name='Calculator', description='Useful for when you need to answer questions about math.', func=<bound method Chain.run of LLMMathChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='Translate a math problem into a expression that can be executed using Python\'s numexpr library. Use the output of running this code to answer the question.\n\nQuestion: ${{Question with math problem.}}\n```text\n${{single line mathematical expression that solves the problem}}\n```\n...numexpr.evaluate(text)...\n```output\n${{Output of running the code}}\n```\nAnswer: ${{Answer}}\n\nBegin.\n\nQuestion: What is 37593 * 67?\n```text\n37593 * 67\n```\n...numexpr.ev

In [21]:
type(mytools[0] )

langchain_community.tools.wikipedia.tool.WikipediaQueryRun

In [22]:
print(response["messages"][-1].content)

최신 뉴스 TOP 5는 다음과 같습니다:

1. **스페인 고속열차 충돌 사고**: 스페인에서 고속열차 간의 충돌로 최소 21명이 사망했습니다.
   
2. **트럼프의 그린란드 발언**: 덴마크 총리가 "유럽은 협박에 굴복하지 않을 것"이라고 말하며, 트럼프의 그린란드 관련 발언에 대한 반응을 보였습니다.

3. **미국 내 군대 배치**: 펜타곤이 미네소타에 1,500명의 군인을 배치할 준비를 하고 있다는 소식이 전해졌습니다.

4. **엘살바도르 교도소의 인권 문제**: 엘살바도르의 악명 높은 교도소에서 수감자들이 겪은 고통스러운 경험이 보도되었습니다.

5. **한국의 아프리카돼지열병**: 한국에서 2개월 만에 첫 아프리카돼지열병 사례가 보고되었습니다.

이 뉴스들은 현재 진행 중인 사건들로, 각 뉴스의 세부 사항은 관련 뉴스 사이트에서 확인할 수 있습니다.
